In [18]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

PROJECT_ROOT = Path.cwd().parent
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

training_df = pd.read_parquet(OUTPUT_DIR / "training_features.parquet")
validation_df = pd.read_parquet(OUTPUT_DIR / "validation_features.parquet")

training_df["prediction_time"] = pd.to_datetime(
    training_df["prediction_time"], utc=True
)
validation_df["prediction_time"] = pd.to_datetime(
    validation_df["prediction_time"], utc=True
)

print("Training shape:", training_df.shape)
print("External holdout shape:", validation_df.shape)


Training shape: (315000, 42)
External holdout shape: (9000, 42)


In [19]:
ACTIVITIES = [
    "open_pr",
    "open_issue",
    "issue_comment",
    "pr_review",
    "pr_review_comment",
    "push",
]

LABEL_COLUMNS = [f"label_{activity}" for activity in ACTIVITIES]
EXCLUDE_COLUMNS = [
    "user_id",
    "github_user_id",
    "login",
    "prediction_time",
] + LABEL_COLUMNS

FEATURE_COLUMNS = [
    column
    for column in training_df.columns
    if column not in EXCLUDE_COLUMNS
]

missing_validation_features = [
    column
    for column in FEATURE_COLUMNS
    if column not in validation_df.columns
]
if missing_validation_features:
    raise ValueError(
        f"Validation data is missing features: {missing_validation_features}"
    )
if training_df[FEATURE_COLUMNS].isna().any().any():
    raise ValueError("Training features contain missing values.")
if validation_df[FEATURE_COLUMNS].isna().any().any():
    raise ValueError("Validation features contain missing values.")

snapshot_dates = sorted(training_df["prediction_time"].unique())
if len(snapshot_dates) < 4:
    raise ValueError("At least four historical snapshots are required.")

TUNE_DATES = snapshot_dates[-4:-2]
FINAL_VAL_DATES = snapshot_dates[-2:]
EXTERNAL_HOLDOUT_DATE = validation_df["prediction_time"].min()
THRESHOLDS = np.arange(0.05, 0.96, 0.05)

if max(TUNE_DATES) >= min(FINAL_VAL_DATES):
    raise ValueError("Tuning dates must precede final-validation dates.")
if max(snapshot_dates) >= EXTERNAL_HOLDOUT_DATE:
    raise ValueError("The external holdout must occur after all training snapshots.")

print("Number of features:", len(FEATURE_COLUMNS))
print("Tuning dates:", TUNE_DATES)
print("Final-validation dates:", FINAL_VAL_DATES)
print("External holdout date:", EXTERNAL_HOLDOUT_DATE)


Number of features: 32
Tuning dates: [Timestamp('2026-10-04 00:00:00+0000', tz='UTC'), Timestamp('2026-10-11 00:00:00+0000', tz='UTC')]
Final-validation dates: [Timestamp('2026-10-18 00:00:00+0000', tz='UTC'), Timestamp('2026-10-25 00:00:00+0000', tz='UTC')]
External holdout date: 2026-11-02 00:00:00+00:00


In [20]:
FIXED_XGB_CONFIG = {
    "name": "fixed",
    "n_estimators": 300,
    "max_depth": 4,
    "learning_rate": 0.05,
    "min_child_weight": 1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "gamma": 0,
    "reg_alpha": 0,
    "reg_lambda": 1,
}

FIXED_HGB_CONFIG = {
    "name": "fixed",
    "learning_rate": 0.05,
    "max_iter": 200,
    "max_leaf_nodes": 15,
    "min_samples_leaf": 20,
    "l2_regularization": 1.0,
}


def calculate_scores(actual, predicted):
    macro_f1 = f1_score(
        actual, predicted, average="macro", zero_division=0
    )
    micro_f1 = f1_score(
        actual, predicted, average="micro", zero_division=0
    )

    result = {
        "macro_f1": macro_f1,
        "micro_f1": micro_f1,
        "combined_score": (macro_f1 + micro_f1) / 2,
    }

    for activity in ACTIVITIES:
        result[f"{activity}_f1"] = f1_score(
            actual[activity], predicted[activity], zero_division=0
        )

    return result


def select_threshold(actual, probabilities):
    best_threshold = 0.5
    best_score = -1.0

    for threshold in THRESHOLDS:
        predicted = (
            np.asarray(probabilities) >= threshold
        ).astype(int)
        score = f1_score(actual, predicted, zero_division=0)

        if score > best_score:
            best_threshold = threshold
            best_score = score

    return float(best_threshold), float(best_score)


def positive_probability(model, features):
    probabilities = model.predict_proba(features)
    classes = list(model.classes_)

    if 1 not in classes:
        return np.zeros(len(features), dtype=float)

    return probabilities[:, classes.index(1)]


In [21]:
def make_model(model_name, labels, config=None):
    unique_classes = np.unique(labels)

    if len(unique_classes) == 1:
        return DummyClassifier(
            strategy="constant",
            constant=int(unique_classes[0]),
        )

    if model_name == "logistic":
        return make_pipeline(
            StandardScaler(),
            LogisticRegression(
                max_iter=3000,
                class_weight="balanced",
                random_state=42,
            ),
        )

    if model_name == "hist":
        config = config or FIXED_HGB_CONFIG
        return HistGradientBoostingClassifier(
            learning_rate=config["learning_rate"],
            max_iter=config["max_iter"],
            max_leaf_nodes=config["max_leaf_nodes"],
            min_samples_leaf=config["min_samples_leaf"],
            l2_regularization=config["l2_regularization"],
            random_state=42,
        )

    if model_name == "xgb":
        config = config or FIXED_XGB_CONFIG
        positive_count = int((labels == 1).sum())
        negative_count = int((labels == 0).sum())

        return XGBClassifier(
            n_estimators=config["n_estimators"],
            max_depth=config["max_depth"],
            learning_rate=config["learning_rate"],
            min_child_weight=config["min_child_weight"],
            subsample=config["subsample"],
            colsample_bytree=config["colsample_bytree"],
            gamma=config["gamma"],
            reg_alpha=config["reg_alpha"],
            reg_lambda=config["reg_lambda"],
            scale_pos_weight=negative_count / positive_count,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=42,
            n_jobs=-1,
        )

    raise ValueError(f"Unknown model: {model_name}")


def fit_model(model_name, features, labels, config=None):
    model = make_model(model_name, labels, config)

    if model_name == "hist" and len(np.unique(labels)) > 1:
        sample_weights = compute_sample_weight(
            class_weight="balanced",
            y=labels,
        )
        model.fit(features, labels, sample_weight=sample_weights)
    else:
        model.fit(features, labels)

    return model


In [22]:
FIXED_XGB_CONFIG = {
    "name": "fixed",
    "n_estimators": 300,
    "max_depth": 4,
    "learning_rate": 0.05,
    "min_child_weight": 1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "gamma": 0,
    "reg_alpha": 0,
    "reg_lambda": 1,
}

FIXED_HGB_CONFIG = {
    "name": "fixed",
    "learning_rate": 0.05,
    "max_iter": 200,
    "max_leaf_nodes": 15,
    "min_samples_leaf": 20,
    "l2_regularization": 1.0,
}


def calculate_scores(actual, predicted):
    macro_f1 = f1_score(
        actual, predicted, average="macro", zero_division=0
    )
    micro_f1 = f1_score(
        actual, predicted, average="micro", zero_division=0
    )

    result = {
        "macro_f1": macro_f1,
        "micro_f1": micro_f1,
        "combined_score": (macro_f1 + micro_f1) / 2,
    }

    for activity in ACTIVITIES:
        result[f"{activity}_f1"] = f1_score(
            actual[activity], predicted[activity], zero_division=0
        )

    return result


def select_threshold(actual, probabilities):
    best_threshold = 0.5
    best_score = -1.0

    for threshold in THRESHOLDS:
        predicted = (
            np.asarray(probabilities) >= threshold
        ).astype(int)
        score = f1_score(actual, predicted, zero_division=0)

        if score > best_score:
            best_threshold = threshold
            best_score = score

    return float(best_threshold), float(best_score)


def positive_probability(model, features):
    probabilities = model.predict_proba(features)
    classes = list(model.classes_)

    if 1 not in classes:
        return np.zeros(len(features), dtype=float)

    return probabilities[:, classes.index(1)]


In [23]:
def tune_thresholds(
    model_name,
    feature_columns,
    configs=None,
):
    pooled_actual = {activity: [] for activity in ACTIVITIES}
    pooled_probabilities = {activity: [] for activity in ACTIVITIES}

    for validation_date in TUNE_DATES:
        train_fold = training_df[
            training_df["prediction_time"] < validation_date
        ]
        validation_fold = training_df[
            training_df["prediction_time"] == validation_date
        ]

        for activity in ACTIVITIES:
            label_column = f"label_{activity}"
            config = configs[activity] if configs is not None else None

            model = fit_model(
                model_name,
                train_fold[feature_columns],
                train_fold[label_column],
                config,
            )
            probabilities = positive_probability(
                model,
                validation_fold[feature_columns],
            )

            pooled_actual[activity].extend(
                validation_fold[label_column].astype(int).tolist()
            )
            pooled_probabilities[activity].extend(
                probabilities.tolist()
            )

    thresholds = {}
    tuning_scores = {}

    for activity in ACTIVITIES:
        threshold, score = select_threshold(
            np.asarray(pooled_actual[activity]),
            np.asarray(pooled_probabilities[activity]),
        )
        thresholds[activity] = threshold
        tuning_scores[activity] = score

    return thresholds, tuning_scores


def evaluate_dates(
    model_name,
    feature_columns,
    thresholds,
    validation_dates,
    configs=None,
):
    rows = []

    for validation_date in validation_dates:
        train_fold = training_df[
            training_df["prediction_time"] < validation_date
        ]
        validation_fold = training_df[
            training_df["prediction_time"] == validation_date
        ]
        predicted = pd.DataFrame(index=validation_fold.index)

        for activity in ACTIVITIES:
            label_column = f"label_{activity}"
            config = configs[activity] if configs is not None else None

            model = fit_model(
                model_name,
                train_fold[feature_columns],
                train_fold[label_column],
                config,
            )
            probabilities = positive_probability(
                model,
                validation_fold[feature_columns],
            )
            predicted[activity] = (
                probabilities >= thresholds[activity]
            ).astype(int)

        actual = validation_fold[LABEL_COLUMNS].copy()
        actual.columns = ACTIVITIES
        result = calculate_scores(actual, predicted)
        result["validation_date"] = validation_date
        rows.append(result)

    return pd.DataFrame(rows)


def summarize_results(name, results):
    return {
        "model": name,
        "mean_macro_f1": results["macro_f1"].mean(),
        "mean_micro_f1": results["micro_f1"].mean(),
        "mean_combined_score": results["combined_score"].mean(),
        "minimum_combined_score": results["combined_score"].min(),
        "maximum_combined_score": results["combined_score"].max(),
    }


#Fixed-model comparison


In [24]:
fixed_xgb_configs = {
    activity: FIXED_XGB_CONFIG for activity in ACTIVITIES
}
fixed_hgb_configs = {
    activity: FIXED_HGB_CONFIG for activity in ACTIVITIES
}

MODEL_SPECS = {
    "Logistic Regression": {
        "model_name": "logistic",
        "configs": None,
    },
    "Histogram Gradient Boosting": {
        "model_name": "hist",
        "configs": fixed_hgb_configs,
    },
    "Fixed XGBoost": {
        "model_name": "xgb",
        "configs": fixed_xgb_configs,
    },
}

validation_results_by_model = {}
thresholds_by_model = {}
comparison_rows = []

for display_name, specification in MODEL_SPECS.items():
    print(f"Running {display_name}...")

    thresholds, _ = tune_thresholds(
        model_name=specification["model_name"],
        feature_columns=FEATURE_COLUMNS,
        configs=specification["configs"],
    )
    results = evaluate_dates(
        model_name=specification["model_name"],
        feature_columns=FEATURE_COLUMNS,
        thresholds=thresholds,
        validation_dates=FINAL_VAL_DATES,
        configs=specification["configs"],
    )

    thresholds_by_model[display_name] = thresholds
    validation_results_by_model[display_name] = results
    comparison_rows.append(summarize_results(display_name, results))

    safe_name = display_name.lower().replace(" ", "_")
    results.to_csv(
        OUTPUT_DIR / f"{safe_name}_final_validation.csv",
        index=False,
    )

fixed_model_comparison = pd.DataFrame(comparison_rows).sort_values(
    "mean_combined_score", ascending=False
).reset_index(drop=True)
fixed_model_comparison.to_csv(
    OUTPUT_DIR / "fixed_model_comparison.csv",
    index=False,
)
fixed_model_comparison


Running Logistic Regression...
Running Histogram Gradient Boosting...
Running Fixed XGBoost...


,model,mean_macro_f1,mean_micro_f1,mean_combined_score,minimum_combined_score,maximum_combined_score
0,Fixed XGBoost,0.415811,0.496483,0.456147,0.409577,0.502717
1,Histogram Gradient Boosting,0.413206,0.496457,0.454831,0.410067,0.499596
2,Logistic Regression,0.390782,0.486741,0.438762,0.391864,0.485659


In [25]:
XGB_CONFIGS = [
    {**FIXED_XGB_CONFIG, "name": "depth_2", "max_depth": 2},
    {**FIXED_XGB_CONFIG, "name": "depth_3", "max_depth": 3},
    {**FIXED_XGB_CONFIG, "name": "depth_4", "max_depth": 4},
    {
        "name": "conservative",
        "n_estimators": 400,
        "max_depth": 3,
        "learning_rate": 0.03,
        "min_child_weight": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "gamma": 0.5,
        "reg_alpha": 0,
        "reg_lambda": 2,
    },
    {
        "name": "regularized",
        "n_estimators": 400,
        "max_depth": 4,
        "learning_rate": 0.03,
        "min_child_weight": 3,
        "subsample": 0.7,
        "colsample_bytree": 0.7,
        "gamma": 0.5,
        "reg_alpha": 0.1,
        "reg_lambda": 3,
    },
]


def tune_activity_specific_settings(
    model_name,
    feature_columns,
    candidate_configs,
):
    settings = {}

    for activity in ACTIVITIES:
        label_column = f"label_{activity}"
        best = {
            "score": -1.0,
            "config": None,
            "threshold": None,
        }

        for config in candidate_configs:
            actual_values = []
            probability_values = []

            for validation_date in TUNE_DATES:
                train_fold = training_df[
                    training_df["prediction_time"] < validation_date
                ]
                validation_fold = training_df[
                    training_df["prediction_time"] == validation_date
                ]

                model = fit_model(
                    model_name,
                    train_fold[feature_columns],
                    train_fold[label_column],
                    config,
                )
                probabilities = positive_probability(
                    model,
                    validation_fold[feature_columns],
                )

                actual_values.extend(
                    validation_fold[label_column].astype(int).tolist()
                )
                probability_values.extend(probabilities.tolist())

            threshold, score = select_threshold(
                np.asarray(actual_values),
                np.asarray(probability_values),
            )

            if score > best["score"]:
                best = {
                    "score": score,
                    "config": config,
                    "threshold": threshold,
                }

        settings[activity] = best
        print(
            activity,
            best["config"]["name"],
            best["threshold"],
            best["score"],
        )

    return settings


In [26]:
best_xgb_settings = tune_activity_specific_settings(
    model_name="xgb",
    feature_columns=FEATURE_COLUMNS,
    candidate_configs=XGB_CONFIGS,
)

tuned_xgb_configs = {
    activity: best_xgb_settings[activity]["config"]
    for activity in ACTIVITIES
}
tuned_xgb_thresholds = {
    activity: best_xgb_settings[activity]["threshold"]
    for activity in ACTIVITIES
}

tuned_xgb_validation = evaluate_dates(
    model_name="xgb",
    feature_columns=FEATURE_COLUMNS,
    thresholds=tuned_xgb_thresholds,
    validation_dates=FINAL_VAL_DATES,
    configs=tuned_xgb_configs,
)

xgb_settings_df = pd.DataFrame([
    {
        "activity": activity,
        "config": best_xgb_settings[activity]["config"]["name"],
        "threshold": best_xgb_settings[activity]["threshold"],
        "tuning_f1": best_xgb_settings[activity]["score"],
    }
    for activity in ACTIVITIES
])

xgb_settings_df.to_csv(
    OUTPUT_DIR / "best_xgb_settings.csv", index=False
)
tuned_xgb_validation.to_csv(
    OUTPUT_DIR / "tuned_xgb_final_validation.csv", index=False
)
xgb_settings_df


open_pr regularized 0.75 0.44759825327510916
open_issue depth_3 0.9 0.49744114636642783
issue_comment regularized 0.8500000000000001 0.4380639868744873
pr_review depth_4 0.8500000000000001 0.42120141342756184
pr_review_comment regularized 0.9 0.3826530612244898
push depth_4 0.45 0.6772037763869736


,activity,config,threshold,tuning_f1
0,open_pr,regularized,0.75,0.447598
1,open_issue,depth_3,0.90,0.497441
2,issue_comment,regularized,0.85,0.438064
3,pr_review,depth_4,0.85,0.421201
4,pr_review_comment,regularized,0.90,0.382653
5,push,depth_4,0.45,0.677204


In [27]:
HGB_CONFIGS = [
    FIXED_HGB_CONFIG,
    {
        "name": "smaller_tree",
        "learning_rate": 0.05,
        "max_iter": 200,
        "max_leaf_nodes": 7,
        "min_samples_leaf": 20,
        "l2_regularization": 1.0,
    },
    {
        "name": "larger_tree",
        "learning_rate": 0.05,
        "max_iter": 200,
        "max_leaf_nodes": 31,
        "min_samples_leaf": 20,
        "l2_regularization": 1.0,
    },
    {
        "name": "conservative",
        "learning_rate": 0.03,
        "max_iter": 300,
        "max_leaf_nodes": 15,
        "min_samples_leaf": 30,
        "l2_regularization": 2.0,
    },
    {
        "name": "regularized",
        "learning_rate": 0.03,
        "max_iter": 300,
        "max_leaf_nodes": 15,
        "min_samples_leaf": 40,
        "l2_regularization": 5.0,
    },
]

best_hgb_settings = tune_activity_specific_settings(
    model_name="hist",
    feature_columns=FEATURE_COLUMNS,
    candidate_configs=HGB_CONFIGS,
)

tuned_hgb_configs = {
    activity: best_hgb_settings[activity]["config"]
    for activity in ACTIVITIES
}
tuned_hgb_thresholds = {
    activity: best_hgb_settings[activity]["threshold"]
    for activity in ACTIVITIES
}

tuned_hgb_validation = evaluate_dates(
    model_name="hist",
    feature_columns=FEATURE_COLUMNS,
    thresholds=tuned_hgb_thresholds,
    validation_dates=FINAL_VAL_DATES,
    configs=tuned_hgb_configs,
)

hgb_settings_df = pd.DataFrame([
    {
        "activity": activity,
        "config": best_hgb_settings[activity]["config"]["name"],
        "threshold": best_hgb_settings[activity]["threshold"],
        "tuning_f1": best_hgb_settings[activity]["score"],
    }
    for activity in ACTIVITIES
])

hgb_settings_df.to_csv(
    OUTPUT_DIR / "best_hgb_settings.csv", index=False
)
tuned_hgb_validation.to_csv(
    OUTPUT_DIR / "tuned_hgb_final_validation.csv", index=False
)
hgb_settings_df


open_pr larger_tree 0.75 0.44555555555555554
open_issue smaller_tree 0.9 0.494949494949495
issue_comment larger_tree 0.8500000000000001 0.4378698224852071
pr_review larger_tree 0.8500000000000001 0.420391061452514
pr_review_comment smaller_tree 0.9500000000000001 0.3886792452830189
push fixed 0.5 0.6771355384786725


,activity,config,threshold,tuning_f1
0,open_pr,larger_tree,0.75,0.445556
1,open_issue,smaller_tree,0.90,0.494949
2,issue_comment,larger_tree,0.85,0.437870
3,pr_review,larger_tree,0.85,0.420391
4,pr_review_comment,smaller_tree,0.95,0.388679
5,push,fixed,0.50,0.677136


In [28]:
tuning_comparison = pd.DataFrame([
    summarize_results(
        "Fixed XGBoost",
        validation_results_by_model["Fixed XGBoost"],
    ),
    summarize_results(
        "Activity-specific tuned XGBoost",
        tuned_xgb_validation,
    ),
    summarize_results(
        "Fixed HGB",
        validation_results_by_model["Histogram Gradient Boosting"],
    ),
    summarize_results(
        "Activity-specific tuned HGB",
        tuned_hgb_validation,
    ),
]).sort_values(
    "mean_combined_score", ascending=False
).reset_index(drop=True)

tuning_comparison.to_csv(
    OUTPUT_DIR / "tuning_comparison.csv", index=False
)
tuning_comparison


,model,mean_macro_f1,mean_micro_f1,mean_combined_score,minimum_combined_score,maximum_combined_score
0,Activity-specific tuned XGBoost,0.422151,0.498886,0.460518,0.413268,0.507769
1,Fixed XGBoost,0.415811,0.496483,0.456147,0.409577,0.502717
2,Fixed HGB,0.413206,0.496457,0.454831,0.410067,0.499596
3,Activity-specific tuned HGB,0.410227,0.498669,0.454448,0.416273,0.492622


In [29]:
REPOSITORY_FEATURES = [
    column
    for column in FEATURE_COLUMNS
    if (
        column.startswith("distinct_repos_")
        or column == "total_historical_repos"
    )
]
ACTIVITY_ONLY_FEATURES = [
    column
    for column in FEATURE_COLUMNS
    if column not in REPOSITORY_FEATURES
]

RELATIONSHIP_FEATURE_SETS = {
    "Activity features only": ACTIVITY_ONLY_FEATURES,
    "Activity + repository features": FEATURE_COLUMNS,
}


def run_fixed_hgb_ablation(feature_sets, experiment_name):
    rows = []

    for feature_set_name, feature_columns in feature_sets.items():
        print(f"Running {feature_set_name}...")

        thresholds, _ = tune_thresholds(
            model_name="hist",
            feature_columns=feature_columns,
            configs=fixed_hgb_configs,
        )
        results = evaluate_dates(
            model_name="hist",
            feature_columns=feature_columns,
            thresholds=thresholds,
            validation_dates=FINAL_VAL_DATES,
            configs=fixed_hgb_configs,
        )

        summary = summarize_results(feature_set_name, results)
        summary["experiment"] = experiment_name
        summary["number_of_features"] = len(feature_columns)
        rows.append(summary)

    return pd.DataFrame(rows)


relationship_results = run_fixed_hgb_ablation(
    RELATIONSHIP_FEATURE_SETS,
    "Relationship feature ablation",
)
relationship_results.to_csv(
    OUTPUT_DIR / "relationship_ablation_results.csv",
    index=False,
)
relationship_results


Running Activity features only...
Running Activity + repository features...


,model,mean_macro_f1,mean_micro_f1,mean_combined_score,minimum_combined_score,maximum_combined_score,experiment,number_of_features
0,Activity features only,0.423272,0.497925,0.460598,0.414129,0.507067,Relationship feature ablation,28
1,Activity + repository features,0.413206,0.496457,0.454831,0.410067,0.499596,Relationship feature ablation,32


In [30]:
TEMPORAL_FEATURE_SETS = {
    "7-day window only": [
        column for column in FEATURE_COLUMNS
        if column.endswith("_7d")
    ],
    "7-day + 30-day windows": [
        column for column in FEATURE_COLUMNS
        if column.endswith(("_7d", "_30d"))
    ],
    "7-day + 30-day + 90-day windows": [
        column for column in FEATURE_COLUMNS
        if column.endswith(("_7d", "_30d", "_90d"))
    ],
    "All windows + recency": [
        column for column in FEATURE_COLUMNS
        if (
            column.endswith(("_7d", "_30d", "_90d"))
            or column.startswith("days_since_last_")
        )
    ],
    "Full feature set": FEATURE_COLUMNS,
}

temporal_results = run_fixed_hgb_ablation(
    TEMPORAL_FEATURE_SETS,
    "Temporal feature ablation",
)
temporal_results.to_csv(
    OUTPUT_DIR / "temporal_ablation_results.csv",
    index=False,
)
temporal_results


Running 7-day window only...
Running 7-day + 30-day windows...
Running 7-day + 30-day + 90-day windows...
Running All windows + recency...
Running Full feature set...


,model,mean_macro_f1,mean_micro_f1,mean_combined_score,minimum_combined_score,maximum_combined_score,experiment,number_of_features
0,7-day window only,0.371789,0.457027,0.414408,0.370854,0.457962,Temporal feature ablation,8
1,7-day + 30-day windows,0.381735,0.486199,0.433967,0.388885,0.479048,Temporal feature ablation,16
2,7-day + 30-day + 90-day windows,0.413459,0.493782,0.453620,0.404344,0.502897,Temporal feature ablation,24
3,All windows + recency,0.418269,0.497213,0.457741,0.410637,0.504845,Temporal feature ablation,30
4,Full feature set,0.413206,0.496457,0.454831,0.410067,0.499596,Temporal feature ablation,32


#External holdout comparison

In [31]:
def evaluate_external_holdout(
    model_name,
    feature_columns,
    thresholds,
    configs=None,
):
    predicted = pd.DataFrame(index=validation_df.index)

    for activity in ACTIVITIES:
        label_column = f"label_{activity}"
        config = configs[activity] if configs is not None else None

        model = fit_model(
            model_name,
            training_df[feature_columns],
            training_df[label_column],
            config,
        )
        probabilities = positive_probability(
            model,
            validation_df[feature_columns],
        )
        predicted[activity] = (
            probabilities >= thresholds[activity]
        ).astype(int)

    actual = validation_df[LABEL_COLUMNS].copy()
    actual.columns = ACTIVITIES
    scores = calculate_scores(actual, predicted)

    return scores, predicted


In [32]:
HOLDOUT_MODEL_SPECS = {
    "Logistic Regression": {
        "model_name": "logistic",
        "thresholds": thresholds_by_model["Logistic Regression"],
        "configs": None,
    },
    "Histogram Gradient Boosting": {
        "model_name": "hist",
        "thresholds": thresholds_by_model[
            "Histogram Gradient Boosting"
        ],
        "configs": fixed_hgb_configs,
    },
    "Fixed XGBoost": {
        "model_name": "xgb",
        "thresholds": thresholds_by_model["Fixed XGBoost"],
        "configs": fixed_xgb_configs,
    },
    "Activity-specific tuned XGBoost": {
        "model_name": "xgb",
        "thresholds": tuned_xgb_thresholds,
        "configs": tuned_xgb_configs,
    },
    "Activity-specific tuned HGB": {
        "model_name": "hist",
        "thresholds": tuned_hgb_thresholds,
        "configs": tuned_hgb_configs,
    },
}

holdout_results = {}
holdout_predictions = {}
holdout_rows = []

for display_name, specification in HOLDOUT_MODEL_SPECS.items():
    print(f"Evaluating {display_name}...")

    scores, predictions = evaluate_external_holdout(
        model_name=specification["model_name"],
        feature_columns=FEATURE_COLUMNS,
        thresholds=specification["thresholds"],
        configs=specification["configs"],
    )

    holdout_results[display_name] = scores
    holdout_predictions[display_name] = predictions
    holdout_rows.append({
        "model": display_name,
        "macro_f1": scores["macro_f1"],
        "micro_f1": scores["micro_f1"],
        "combined_score": scores["combined_score"],
    })

complete_holdout_comparison = pd.DataFrame(holdout_rows).sort_values(
    "combined_score", ascending=False
).reset_index(drop=True)
complete_holdout_comparison.to_csv(
    OUTPUT_DIR / "complete_holdout_model_comparison.csv",
    index=False,
)
complete_holdout_comparison


Evaluating Logistic Regression...
Evaluating Histogram Gradient Boosting...
Evaluating Fixed XGBoost...
Evaluating Activity-specific tuned XGBoost...
Evaluating Activity-specific tuned HGB...


,model,macro_f1,micro_f1,combined_score
0,Histogram Gradient Boosting,0.383905,0.364657,0.374281
1,Activity-specific tuned XGBoost,0.384949,0.352657,0.368803
2,Activity-specific tuned HGB,0.366094,0.361467,0.363781
3,Fixed XGBoost,0.376694,0.342998,0.359846
4,Logistic Regression,0.334969,0.302574,0.318772


In [33]:
holdout_activity_rows = []

for display_name, scores in holdout_results.items():
    for activity in ACTIVITIES:
        holdout_activity_rows.append({
            "model": display_name,
            "activity": activity,
            "f1": scores[f"{activity}_f1"],
        })

holdout_activity_comparison = (
    pd.DataFrame(holdout_activity_rows)
    .pivot(index="activity", columns="model", values="f1")
    .reset_index()
)
holdout_activity_comparison.to_csv(
    OUTPUT_DIR / "holdout_activity_model_comparison.csv",
    index=False,
)
holdout_activity_comparison


model,activity,Activity-specific tuned HGB,Activity-specific tuned XGBoost,Fixed XGBoost,Histogram Gradient Boosting,Logistic Regression
0,issue_comment,0.393862,0.395122,0.396088,0.345238,0.359420
1,open_issue,0.413793,0.414986,0.392962,0.419825,0.343643
2,open_pr,0.395387,0.402556,0.350181,0.401961,0.320939
3,pr_review,0.472340,0.469880,0.469880,0.479675,0.385675
4,pr_review_comment,0.220472,0.352332,0.376238,0.356021,0.340136
5,push,0.300709,0.274818,0.274818,0.300709,0.260000


#Final selected model

In [34]:
selected_model_name = "Histogram Gradient Boosting"
selected_thresholds = thresholds_by_model[selected_model_name]

selected_thresholds_df = pd.DataFrame([
    {
        "activity": activity,
        "threshold": selected_thresholds[activity],
    }
    for activity in ACTIVITIES
])

selected_config_df = pd.DataFrame([
    FIXED_HGB_CONFIG
])

selected_thresholds_df.to_csv(
    OUTPUT_DIR / "selected_hgb_thresholds.csv",
    index=False,
)
selected_config_df.to_csv(
    OUTPUT_DIR / "selected_hgb_config.csv",
    index=False,
)

print("Selected model:", selected_model_name)
selected_thresholds_df


Selected model: Histogram Gradient Boosting


,activity,threshold
0,open_pr,0.75
1,open_issue,0.90
2,issue_comment,0.90
3,pr_review,0.85
4,pr_review_comment,0.90
5,push,0.50
